# CS918 Assignment 1

## Part A: Text Preprocessing

The aim of this stage is to make the text “smoother” and homogenize certain tokens. This will allow the language model we train later to be grounded in meaningful semantic units instead of formatting artifacts.

### 1. Normalization (Lowercase)

We lowercase all text so that the model will not treat words like "The" and "the" as two different tokens. This will signiﬁcantly decrease the size of vocabulary ( ), and avoid data sparseness.

### 2. Regex Cleaning (Noise Removal)

We make use of Regular Expressions with the below rules:

(d) URL Removal: URLs (e.g., https://...) are non-semantic and highly variable. They need to be stripped as well since they are puncutation that the other rules will kill.

(a)Non-Alphabetic Filtering: The punctuation and special symbols are removed, the letters and numbers remaining only to consider the core text.

(c)Digit Filtering: We remove numbers as they are often very numerically specific to a single report, but we keep the alphanumeric strings since they carry technical or time-specific semantics.

(b)Length filtration : Words of length 1 are usually punctuation remainders (in this assignment except 'a' and 'i', but they're also often removed anyway).

### 3. Lemmatization

Lemmatization is unlike simple stemming,which just chops off word endings, because it creates an actual word, the lemma, which means that roots are converted to words of some kind.

Example: running run, better good, mice mouse.

Lemmatisation Lemmatization allows us to group the various inflected forms of a word together because words appearing in our Trigram model will have more statistical evidence for meaning.

### Note for myself

Lemmatisation V.S Stemming

Lemmatisation : transforming words

Stemming : roughly chopping off suffixes 


In [17]:
import json
import re
import math
import nltk
from nltk.stem import WordNetLemmatizer
from collections import Counter, defaultdict

# Part a
nltk.download('wordnet')
lemmatizer = WordNetLemmatizer()

def partA(text):
    # (d) Remove URLs first!
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    
    # Lowercase (as requested)
    text = text.lower()
    
    # (a) keep only a-z, 0-9 and spaces
    text = re.sub(r'[^a-z0-9\s]', '', text)
    
    tokens = text.split()
    
    # (c) Remove pure digits AND (b) Remove 1-char words
    cletok = []
    for t in tokens:
        # (c): if it's all digits, skip it
        if t.isdigit():
            continue
        # (b): if length is 1, skip it
        if len(t) <= 1:
            continue
        
        # If it passes,  lemmatize and keep it
        cletok.append(lemmatizer.lemmatize(t))
        all
    return cletok

#  Data Loading 
allstories = []
jsonlpath = 'signal-news1/signal-news1.jsonl' 

print("Processing... go to Part B")
with open(jsonlpath, 'r') as f:
    for line in f:
        data = json.loads(line)
        allstories.append(partA(data['content']))

[nltk_data] Downloading package wordnet to
[nltk_data]     /dcs/pg25/u5693885/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Processing... go to Part B



#### **Why this Order:**

We have the order (d) Lower case letters (a) (c)(b).

1. Step (d): Remov e URLs:URLs such as https://...',www. *) must be stripped first. URLs have punctuation (such as /, . , and :).

   If we were to remove punctuation (Rule a) before URLs, for example a URL such as https://warwick.ac.uk would be tokenized into httpswarwickacuk`.

   This prevents the model from recognizing it as a link and filtering, leaving "garbage" tokens shared in our vocabulary ().

3. Step :Lowercasing All the text is converted to lowercase, so that "The" and "the" are considered as one word instead of two words, leading to the reduction in the size of our vocabulary.

4. Step (a) :No.AlphaNumeric Filtering: We eliminate punctuation and special characters, retaining solely letters, numbers and whitespaces.

   This step is necessary for having rules (c) and (b), in that it isolates words from punctuation around them (e.g., turning "5." into "5"`).

6. Step (c) :Digit Filtering: After punctuation has been eliminated we can recognize "pure" numerals, e.g., "100", "2026".

   We filter these out as they have little empirical value for language modeling.

   But we preserve them among alphanumeric strings (e.g., "5pm", "4G") since it has temporal or technical meanings.

8. Step (b) :Length Filtration: Finally, we eliminate words of length one.

   After punctuation has been stripped, many segments (like an isolated 's' or 't') are left over.

   The last step of length filtering is a “final sweep”, where we make sure that there are no non-words in our corpus.




## Part B

In this section, we perform statistical analysis on the preprocessed corpus. The goal is to understand the distribution of words, identify Part B

In this section we do some statistical analysis of the preprocessed corpus. The target here is to analyze word distribution, find out common phrases(trigrams) and sentiment-check the news stories.

### 1. Corpus Statistics (N and V)

N (Number of Tokens): Size of the corpus in words after preprocessing. That’s our data dimension.

V (Vocabulary Size): The quantity of unique words (types). This is the variation captured by the corpus.

The number of word pairs and also turn out to be Zipf distributed, with the majority of them being small in count, as predicted by Zipf's Law.

### 2. Trigram Generation

A Trigram is an ordered list of three, consecutive tokens. These sequences are obtained by a sliding window per story.

Why Trigrams? They contain more information than unigrams (single words) or bigrams (word pairs).

Application: The top 25 trigrams represent the dominant topics of the corpus (such as financial reporting or analysis of stock market).

### 3. Sentiment Analysis (Lexicon-based)

Then, the news stories are sentiment-annotated by employing a Lexicon-based approach. By contrast with pre-defined lists of positive and negative words, by comparing our corpus words:

Word Counts: We add up all the positive and negative words over the entire data set.

Story wise classification: We compare the counting of positive-nagative words of a news to determine if it is overall "Positive" or vice versa for every single piece of news story.

### 4. Implementation Logic

Set Lookup: We keep the sentiment lexicons as Python set objects. This enables O(1) complexity lookups, necessary for processing english text with millions of tokens.

Flattening: We flatten the stories list as an single stream of tokens for computing global statistics, but un-flatten to keep them separately while generating trigrams and avoid "cross-story" artifacts.


In [18]:
from collections import Counter

# Part b

# 1. Compute N  and V 
# Flatten the list of stories into a single list of all tokens
alltok = [token for story in allstories for token in story]

N = len(alltok)          # Total number of tokens
V = len(set(alltok))     # Number of unique tokens 

print(f"Total Tokens (N): {N}")
print(f"Vocabulary Size (V): {V}")

# 2. List the top 25 trigrams based on occurrences
trigrams = []
for story in allstories:
    # Only stories with at least 3 words can form a trigram
    if len(story) >= 3:
        for i in range(len(story) - 2):
            # Create a tuple of 3 consecutive words
            trigram = (story[i], story[i+1], story[i+2])
            trigrams.append(trigram)

# Use Counter to find the 25 most common trigrams
top25tri = Counter(trigrams).most_common(25)

print("\nTop 25 Trigrams:")
for tri, count in top25tri:
    print(f"{tri}: {count}")

# 3. Positive and Negative
# Defining the file paths
pos_path = 'signal-news1/opinion-lexicon-English/positive-words.txt'
neg_path = 'signal-news1/opinion-lexicon-English/negative-words.txt'

#Load the sentiment words into sets so they can be looked up fast in O(1) time.
#We discard lines beginning with ';' (comments) and empty lines.
#Encoding type to fit special characters in the lexicon files using ISO-8859-1.
pos_set = set(line.strip() for line in open(pos_path, encoding='ISO-8859-1') 
              if line.strip() and not line.startswith(';'))
neg_set = set(line.strip() for line in open(neg_path, encoding='ISO-8859-1') 
              if line.strip() and not line.startswith(';'))

totalposcount = 0
totalnegcount = 0

# 4. Count stories with more positive/negative words
stormorepos = 0
stormoreneg = 0

for story in allstories:
    # Count how many positive/negative words appear in this specific story
    nowpos = sum(1 for word in story if word in pos_set)
    nowneg = sum(1 for word in story if word in neg_set)
    
    # Add to the global running total for the entire corpus
    totalposcount += nowpos
    totalnegcount += nowneg
    
    # Compare counts for the current story (Rule: ties are not counted)
    if nowpos > nowneg:
        stormorepos += 1
    elif nowneg > nowpos:
        stormoreneg += 1

print(f"\nTotal Positive Words in Corpus: {totalposcount}")
print(f"Total Negative Words in Corpus: {totalnegcount}")
print(f"Stories with more Positive words: {stormorepos}")
print(f"Stories with more Negative words: {stormoreneg}")

Total Tokens (N): 5695431
Vocabulary Size (V): 124530

Top 25 Trigrams:
('one', 'of', 'the'): 2434
('on', 'share', 'of'): 2093
('on', 'the', 'stock'): 1567
('a', 'well', 'a'): 1424
('in', 'research', 'report'): 1415
('in', 'research', 'note'): 1373
('for', 'the', 'quarter'): 1221
('the', 'united', 'state'): 1221
('average', 'price', 'of'): 1193
('research', 'report', 'on'): 1177
('research', 'note', 'on'): 1138
('share', 'of', 'the'): 1132
('the', 'end', 'of'): 1130
('in', 'report', 'on'): 1124
('earnings', 'per', 'share'): 1121
('cell', 'phone', 'plan'): 1073
('phone', 'plan', 'detail'): 1070
('according', 'to', 'the'): 1063
('of', 'the', 'company'): 1057
('buy', 'rating', 'to'): 1016
('appeared', 'first', 'on'): 995
('moving', 'average', 'price'): 995
('day', 'moving', 'average'): 993
('price', 'target', 'on'): 981
('part', 'of', 'the'): 934

Total Positive Words in Corpus: 170762
Total Negative Words in Corpus: 129763
Stories with more Positive words: 10824
Stories with more Negativ



#### **Technical Implementation:**

* **Encoding:** We use `ISO-8859-1` to load the lexicon files. This legacy encoding ensures compatibility with older text files that might contain non-UTF-8 characters.
* **Efficiency:** Lexicons are stored in **Python Sets**. This allows for  lookup time, enabling the program to check millions of words against the dictionary in seconds.
* **Classification Logic:** 1. We count positive and negative word occurrences per story.
2. We compare the counts: if , the story is marked as positive (and vice versa).
3. **Ties are ignored** as per the assignment requirements.

---

###  Summary of Results

Based on our execution:

* **Total Tokens (N):** 5,695,431
* **Vocabulary (V):** 124,530
* **Sentiment Bias:** The corpus contains more "Positive-leaning" stories (10,824) than "Negative-leaning" ones (6,395), suggesting a generally optimistic tone in the financial news reporting.


## Part C: Language Modeling and Evaluation

For the last part of this homework, we build a Statistical Language Model on Trigrams. The intent is to train the model on part of the news corpus, and see if it can predict unseen text.

### 1. Training and Data Splitting

We divide the preprocessed allstories into two sets as follows:

**Training Set**: The first 16,000 stories used to calculate frequency counts.

**Test Set**: The rest of the stories after (row 16,001 and below) used to evaluate the performance.

We can hold these counts in a defaultdict, for Trigrams $(w_1, w_2, w_3)$ and Bigrams $(w_1, w_2)$ to assist us to compute conditional probabilities.

### 2. Sentence Generation (Greedy Search)

Given the seed bigram “is this”, the model predicts the most probable next word, based on which word occurs most often in the training set. This procedure is iteratively performed to generate sentence of 10 words. Greedy Search: this is called Greedy search because once the model selects its current best candidate it never goes back to re-evaluate it.

### 3. Perplexity and Laplace Smoothing

We evaluate how well the model can predict the test data with Perplexity (PP).

Sparsity: Many word all also be zero. combinations in the Test Set do not appear in the training data, so P(word_i | category_k) will many Of course.

Laplace (Add-one) Smoothing: We increment 1 to each trigram count and add the size of vocabulary $V$ at denominator. This guarantees no zero probability and ensures a model to be more robust.

Log-Space Calculation: All probabilities are added together in log2 space to avoid numerical underflow.

The final Perplexity score is the "average number of options" that it feels it has as choices to make when predicting the next word. The lower its value, the better.

In [22]:
import math
from collections import Counter, defaultdict

# Part c

# 1. Split the data: First 16,000 for training, the rest for testing
train = allstories[:16000]
test = allstories[16000:]

# We use a nested dictionary for faster training: { (w1, w2): {w3: count} }
tricounts = defaultdict(lambda: defaultdict(int))
bigcounts = Counter()

print("Training Trigram Language Model on 16,000 stories")
for story in train:
    for i in range(len(story) - 2):
        w1, w2, w3 = story[i], story[i+1], story[i+2]
        tricounts[(w1, w2)][w3] += 1
        bigcounts[(w1, w2)] += 1

#  2. Sentence Generation 
print("Generating sentence starting with 'is this'")
currbigram = ('is', 'this')
gensentence = ['is', 'this']

for _ in range(8):
    # Get all candidate words that followed this bigram in training
    nextwords = tricounts[currbigram]
    
    if not nextwords:
        break # Stop if the bigram was never seen
    
    # Pick the word with the highest count
    most_likely_next_word = max(nextwords, key=nextwords.get)
    gensentence.append(most_likely_next_word)
    
    # Shift the window: (w1, w2, w3) -> (w2, w3)
    currbigram = (currbigram[1], most_likely_next_word)

print(f"Result: {' '.join(gensentence)}")

#  3. Perplexity Calculation
# V is the vocabulary size from Part b (len(set(all_tokens)))
V = len(set(token for story in allstories for token in story))

log_prob_sum = 0
trigram_count_test = 0

print(f"Calculating Perplexity on test set (V={V})")
for story in test:
    if len(story) < 3:
        continue
    for i in range(len(story) - 2):
        w1, w2, w3 = story[i], story[i+1], story[i+2]
        
        # Get counts (defaults to 0 if not found)
        c_tri = tricounts[(w1, w2)][w3]
        c_bi = bigcounts[(w1, w2)]
        
        # Laplace Smoothing formula
        prob = (c_tri + 1) / (c_bi + V)
        
        # Accumulate log probability
        log_prob_sum += math.log2(prob)
        trigram_count_test += 1

# Perplexity Final Formula
if trigram_count_test > 0:
    avg_log_prob = log_prob_sum / trigram_count_test
    perplexity = 2 ** (-avg_log_prob)
    print(f"\n Final Result ")
    print(f"Perplexity: {perplexity:.2f}")
else:
    print("Error: No test data processed.")

Training Trigram Language Model on 16,000 stories
Generating sentence starting with 'is this'
Result: is this the company ha market capitalization of billion and
Calculating Perplexity on test set (V=124530)

 Final Result 
Perplexity: 47219.32
